In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import NL_Trumelan
import glob
import numpy as np


In [9]:
driver = "vGAT"
responder = "eOPN3_ATR" #for any extra names that dont fit, just change the file name temporarily
illum_duration = 30
time = str(illum_duration) + "s"

genotypefilename = driver + " x " + responder + "_" + time
drivercontrol_name = "w1118 x " + driver + "_" + time
respondercontrol_name = "w1118 x " + responder  + "_" + time

laptop = "C:\\Users\\lnico"
workcomp = "C:\\Users\\User"
homecomp = "D:"
base_path = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data\\TruMelan\\Reformatted\\"  

titledpath = homecomp
specifiedpath = titledpath + base_path

bin_size = 0.5  # seconds for heatmap resampling
framerate = 10  
illum_start = 60  # illumination start (seconds)

max_time = 3600

exp_path = specifiedpath + genotypefilename 
drivercontrol_path = specifiedpath + drivercontrol_name 
respondercontrol_path = specifiedpath + respondercontrol_name 

#change here
fileofinterest =exp_path
fileofinterestname = genotypefilename 

print(exp_path )
print(genotypefilename )

D:\ACC Lab Dropbox\ACC Lab\Nicole Lee\eOPN3 manuscript\Data\TruMelan\Reformatted\vGAT x eOPN3_ATR_30s
vGAT x eOPN3_ATR_30s


## IDX to csv

In [9]:
input_folder = r"D:\ACC Lab Dropbox\ACC Lab\Leif\2026Janexpts\trumelan data\20260220\vGAT x eOPN3_ATR_5s"

custom_name = None  #put to None, if same exact name as folder. output will be in the format of: date {insert custom name}_fly number
NL_Trumelan.idx_to_csv(input=input_folder, output=input_folder, custom_name=custom_name)

100%|██████████| 164919/164919 [00:00<00:00, 231223.60it/s]


## Generating total processing file

In [10]:
def savingtocsvfiles(filepath, outputpath = None, name = None, time_filter = None, max_time = None, bin_size = None):
    df_combined = NL_Trumelan.process_genotype_data(filepath, time_filter= time_filter, max_time=max_time, bin_size = bin_size)
    df_combined.to_csv(outputpath + name +  "_max" + str(max_time) + "s.csv" )

In [11]:
outputfolder = titledpath + "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\Trumelan\\2. Processed\\" 

#change to corresponding control

savingtocsvfiles(fileofinterest, outputpath = outputfolder, name = fileofinterestname + "_batch1", time_filter=time + "_1", max_time=3600, bin_size = 0.5)
#savingtocsvfiles(drivercontrol_path, outputpath = outputfolder, name = drivercontrol_name, time_filter=time, max_time=3600, bin_size = 0.5)
#savingtocsvfiles(respondercontrol_path, outputpath = outputfolder, name = respondercontrol_name , time_filter=time, max_time=3600, bin_size = 0.5)

Time filter '30s_1' applied: 24 files found


## Generating total processing file WITH QC FILTER

In [ ]:

def qc_check_baseline_activity(df, baseline_end=60):
    """
    Quality control check for baseline activity (pre-stimulus period)
    Returns: (pass_status, fail_reason)
    """
    baseline_df = df[df.index <= baseline_end].copy()
    
    activity = baseline_df["Activity Level"]
    speed = baseline_df["Speed_Av_mm_per_sec"]
    
    # CHECK 1: Overall Baseline Activity (entire 0-60s period)
    # Mean Activity Level > 10 pixels OR Mean Speed >= 1 mm/s
    mean_activity = activity.mean()
    mean_speed = speed.mean()
    
    if not (mean_activity > 10 or mean_speed >= 1):
        return False, f"Low baseline activity (Activity: {mean_activity:.1f}px, Speed: {mean_speed:.3f}mm/s)"
    
    # CHECK 2a: Binned Activity Check (6 bins of 10s each)
    # Each bin must have: Mean Activity Level > 10 OR Mean Speed >= 1
    bin_size = 10  # seconds
    num_bins = int(baseline_end / bin_size)
    
    for i in range(num_bins):
        bin_start = i * bin_size
        bin_end = (i + 1) * bin_size
        bin_df = baseline_df[(baseline_df.index > bin_start) & (baseline_df.index <= bin_end)]
        
        if len(bin_df) == 0:
            continue
        
        bin_mean_activity = bin_df["Activity Level"].mean()
        bin_mean_speed = bin_df["Speed_Av_mm_per_sec"].mean()
        
        if not (bin_mean_activity > 10 or bin_mean_speed >= 1):
            return False, f"Inactive bin {bin_start}-{bin_end}s (Activity: {bin_mean_activity:.1f}px, Speed: {bin_mean_speed:.3f}mm/s)"
    
    # CHECK 2b: No Steep Decline
    # Compare last 20s (40-60s) to first 20s (0-20s)
    # Either Activity Level OR Speed must maintain >= 50%
    first_20s = baseline_df[baseline_df.index <= 20]
    last_20s = baseline_df[(baseline_df.index > 40) & (baseline_df.index <= 60)]
    
    if len(first_20s) > 0 and len(last_20s) > 0:
        first_mean_activity = first_20s["Activity Level"].mean()
        last_mean_activity = last_20s["Activity Level"].mean()
        first_mean_speed = first_20s["Speed_Av_mm_per_sec"].mean()
        last_mean_speed = last_20s["Speed_Av_mm_per_sec"].mean()
        
        activity_ratio = (last_mean_activity / first_mean_activity) if first_mean_activity > 0 else 0
        speed_ratio = (last_mean_speed / first_mean_speed) if first_mean_speed > 0 else 0
        
        if not (activity_ratio >= 0.5 or speed_ratio >= 0.5):
            return False, f"Activity decline (Activity ratio: {activity_ratio:.2f}, Speed ratio: {speed_ratio:.2f})"
    
    return True, ""

In [18]:
def process_genotype_with_qc(filepath, time_filter=None, max_time=None, bin_size=None, qc_filter=True):
    """
    Process genotype data with QC filtering
    Returns the processed dataframe for viewing before saving
    """
    
    # Find CSV files with time filter
    time_pattern = f"_{time_filter}_"
    csv_files = [f for f in glob.glob(os.path.join(filepath, "*.csv")) if time_pattern in os.path.basename(f)]
    print(f"Time filter '{time_filter}' applied: {len(csv_files)} files found")
    
    # Process each file
    all_dataframes = []
    qc_passed = 0
    qc_failed = 0
    failed_flies = []
    
    for i, file in enumerate(csv_files, start=1):
        # Read CSV file
        df = pd.read_csv(file, usecols=["Timestamp (ms)", "Activity Level", "Speed_Av_mm_per_sec"])
        df["Time (s)"] = (df["Timestamp (ms)"] - df["Timestamp (ms)"].iloc[0]) / 1000
        df = df.set_index("Time (s)")
        df = df[df.index <= max_time]
        
        # Apply QC filter
        if qc_filter:
            qc_pass, qc_reason = qc_check_baseline_activity(df, baseline_end=60)
            if not qc_pass:
                qc_failed += 1
                failed_flies.append((os.path.basename(file), qc_reason))
                continue  # Skip this fly
            else:
                qc_passed += 1
        
        # Create individual dataframe for this fly
        fly_df = pd.DataFrame(index=df.index)
        fly_df[f"ActivityLevel_{qc_passed}"] = df["Activity Level"]
        fly_df[f"Speed_{qc_passed}"] = df["Speed_Av_mm_per_sec"]
        all_dataframes.append(fly_df)
    
    # Print QC summary
    if qc_filter:
        print(f"\n=== QC Summary ===")
        print(f"Total flies: {len(csv_files)}")
        print(f"Passed QC: {qc_passed}")
        print(f"Failed QC: {qc_failed}")
        if qc_failed > 0:
            print(f"\nFailed flies:")
            for fly_name, reason in failed_flies:
                print(f"  - {fly_name}: {reason}")
        print(f"==================\n")
    
    # Concatenate and bin
    combined_df = pd.concat(all_dataframes, axis=1)
    binned_df = combined_df.groupby((combined_df.index // bin_size) * bin_size).mean()
    
    # Log10 transform Activity Level columns
    activity_cols = [col for col in binned_df.columns if col.startswith("ActivityLevel")]
    for col in activity_cols:
        fly_num = col.split("_")[1]
        log_col_name = f"ActivityLevelLog10_{fly_num}"
        binned_df[log_col_name] = np.log10(binned_df[col] + 1)
        binned_df.drop(columns=[col], inplace=True)
    
    return binned_df

In [28]:
# Process data WITH QC and view before saving
df_combined_qc = process_genotype_with_qc(
    filepath=fileofinterest, 
    time_filter=time, 
    max_time=3600, 
    bin_size=0.5, 
    qc_filter=True
)

# View the dataframe
df_combined_qc

Time filter '5s' applied: 24 files found

=== QC Summary ===
Total flies: 24
Passed QC: 18
Failed QC: 6

Failed flies:
  - 20250519 elav x ACR_5s_12.csv: Activity decline (Activity ratio: 0.09, Speed ratio: 0.18)
  - 20250519 elav x ACR_5s_16.csv: Activity decline (Activity ratio: 0.28, Speed ratio: 0.31)
  - 20250519 elav x ACR_5s_25.csv: Inactive bin 10-20s (Activity: 0.2px, Speed: 0.014mm/s)
  - 20250519 elav x ACR_5s_3.csv: Activity decline (Activity ratio: 0.37, Speed ratio: 0.15)
  - 20250519 elav x ACR_5s_5.csv: Activity decline (Activity ratio: 0.49, Speed ratio: 0.41)
  - 20250519 elav x ACR_5s_6.csv: Activity decline (Activity ratio: 0.28, Speed ratio: 0.27)



,Speed_1,Speed_2,Speed_3,Speed_4,Speed_5,Speed_6,Speed_7,Speed_8,Speed_9,Speed_10,...,ActivityLevelLog10_9,ActivityLevelLog10_10,ActivityLevelLog10_11,ActivityLevelLog10_12,ActivityLevelLog10_13,ActivityLevelLog10_14,ActivityLevelLog10_15,ActivityLevelLog10_16,ActivityLevelLog10_17,ActivityLevelLog10_18
Time (s),,,,,,,,,,,,,,,,,,,,,
0.0,2.813579,3.002536,4.462052,14.440083,8.409548,1.776293,0.047451,7.865957,9.994530,6.834748,...,3.759517,3.840608,3.883241,3.734288,3.837513,3.524630,3.682741,3.665975,3.710896,3.800786
0.5,8.072066,0.780711,0.276778,5.199132,10.543664,5.166692,0.064455,4.497954,8.869427,4.819289,...,3.523980,3.575396,3.712380,3.350403,3.560791,3.079832,3.316264,2.414973,3.398842,3.510089
1.0,10.153714,4.473046,0.220801,6.216506,12.525926,7.459051,0.629466,1.152876,6.717802,6.893678,...,3.382341,3.087213,3.418235,3.059563,3.685795,3.223028,2.596377,2.296665,3.390441,3.697595
1.5,11.270784,4.907982,0.202625,2.198794,8.274631,8.748639,6.372932,3.855856,1.408427,0.512137,...,3.356676,2.806044,3.502891,3.025961,3.598068,2.897077,3.410136,2.379849,3.366273,3.482359
2.0,11.140459,1.354335,0.996415,0.369360,0.587249,6.640434,9.344376,7.167881,0.471338,0.173905,...,3.274620,2.649919,3.523278,3.073352,3.486600,2.365113,3.355605,2.314289,3.396304,3.486317
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3597.5,3.108081,4.974454,5.708712,1.088304,0.750154,6.216148,5.777153,2.931664,2.978127,0.032246,...,3.356485,3.205367,3.654215,3.499989,2.806858,2.083503,3.516377,3.029059,2.945567,3.505774
3598.0,4.934230,4.725210,6.765297,0.303056,1.233806,6.503835,8.179433,5.471795,2.414191,1.208776,...,3.274112,3.456761,3.645638,3.500483,2.896085,1.900913,3.403361,3.084433,2.538322,3.586069
3598.5,4.886350,3.707258,7.970109,0.020240,1.092879,6.624208,7.748328,1.772180,2.934652,2.612198,...,3.174583,3.576226,3.567943,3.511643,2.621799,1.641474,3.176149,3.480697,2.824906,3.608098


In [29]:
# Save QC-filtered data to CSV (only run after viewing above)
outputfolder = homecomp + "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\Trumelan\\2. Processed\\" 
output_file = outputfolder + fileofinterestname + "_" + time + "_max" + str(3600) + "s_QC.csv"

df_combined_qc.to_csv(output_file)
print(f"Saved to: {output_file}")

Saved to: C:\Users\lnico\ACC Lab Dropbox\ACC Lab\Nicole Lee\eOPN3 manuscript\Data compilation\Trumelan\2. Processed\elav x ACR_5s_max3600s_QC.csv
